# TONTOUMA-BOT — Benchmark de la chaîne complète : Image/PDF → Texte → Wolof → Audio

Ce notebook mesure les performances à **chaque étage** (OCR, traduction, TTS) puis la **performance globale du pipeline** — pas seulement le meilleur modèle isolé, mais la meilleure *combinaison*.

```
Image/PDF → OCR → Texte français → Traduction FR→WO → Texte Wolof → TTS → Audio Wolof
              ↓              ↓                              ↓
         CER/WER/F1    BLEU/chrF/TER/COMET          MOS/WER audio/SNR/RTF
                              ↓
                  Performance globale : latence totale, taux de réussite,
                  RAM/VRAM, débit
```

## Étages benchmarkés

| Étage | Modèles comparés |
|---|---|
| OCR | PaddleOCR, EasyOCR, Tesseract, docTR |
| Traduction FR→WO | Lahad-NLLB, cibfaye-NLLB, NLLB général |
| TTS Wolof | XTTS-v2-Wolof, Oolel-Voices, SpeechT5-Wolof |

### Installation (une seule fois)

```bash
pip install paddleocr paddlepaddle easyocr pytesseract python-doctr pypdfium2 \
            transformers torch sacrebleu unbabel-comet jiwer soundfile librosa \
            coqui-tts parler-tts psutil pandas matplotlib huggingface_hub --break-system-packages
```

Sur Linux, Tesseract nécessite aussi le binaire système : `apt install tesseract-ocr tesseract-ocr-fra`

⚠️ Ce notebook charge potentiellement **10 modèles** (4 OCR + 3 traduction + 3 TTS). Prévois du temps, de la RAM et idéalement un GPU. Un mode `MODE_RAPIDE` permet de limiter le nombre de combinaisons testées en bout de chaîne.

## 1. Configuration générale

In [ ]:
import io
import re
import time
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import soundfile as sf
import torch

warnings.filterwarnings("ignore")

MODELES_OCR = {
    "paddleocr": "PaddleOCR",
    "easyocr": "EasyOCR",
    "tesseract": "Tesseract",
    "doctr": "docTR",
}

MODELES_FR_WO = {
    "lahad": "Lahad/nllb200-francais-wolof",
    "cibfaye": "cibfaye/nllb-fr-wo",
    "nllb_general": "facebook/nllb-200-distilled-600M",
}

MODELES_TTS = {
    "xtts_wolof": "galsenai/xTTS-v2-wolof",
    "oolel_voices": "Laurentmd5/Oolel-Voices",
    "speecht5_wolof": "bilalfaye/speecht5-wolof",
}

ASR_MODEL_PATH = "./wolof-whisper-small-lora"  # pour le WER audio du TTS
ASR_BASE_MODEL = "openai/whisper-small"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE_RATE = 16000

DATA_DIR = Path("data_pipeline")          # images/PDF + ground truth OCR
DATA_DIR.mkdir(exist_ok=True)
AUDIO_DIR = Path("audios_pipeline")
AUDIO_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path("resultats_pipeline")
RESULTS_DIR.mkdir(exist_ok=True)
MEILLEUR_PIPELINE_DIR = Path("meilleur_pipeline")

MODE_RAPIDE = True  # limite les combinaisons testées en bout de chaîne (mets False pour tout tester)

print(f"Device utilisé : {DEVICE.upper()}")
print(f"OCR à comparer         : {list(MODELES_OCR.keys())}")
print(f"Traduction à comparer   : {list(MODELES_FR_WO.keys())}")
print(f"TTS à comparer          : {list(MODELES_TTS.keys())}")


## 2. Étage 1 — OCR : chargement des 4 moteurs

Chaque moteur a sa propre API — chargement individuel protégé par `try/except`.

In [ ]:
ocr_engines = {}
echecs_ocr = {}

# --- PaddleOCR ---
try:
    from paddleocr import PaddleOCR
    ocr_engines["paddleocr"] = PaddleOCR(
        lang="fr", use_doc_orientation_classify=True,
        use_doc_unwarping=True, use_textline_orientation=True,
    )
    print("✅ paddleocr chargé")
except Exception as e:
    echecs_ocr["paddleocr"] = str(e)
    print(f"❌ paddleocr : {e}")

# --- EasyOCR ---
try:
    import easyocr
    ocr_engines["easyocr"] = easyocr.Reader(["fr"], gpu=(DEVICE == "cuda"))
    print("✅ easyocr chargé")
except Exception as e:
    echecs_ocr["easyocr"] = str(e)
    print(f"❌ easyocr : {e}")

# --- Tesseract ---
try:
    import pytesseract
    pytesseract.get_tesseract_version()  # vérifie que le binaire système est bien installé
    ocr_engines["tesseract"] = pytesseract
    print("✅ tesseract détecté")
except Exception as e:
    echecs_ocr["tesseract"] = str(e)
    print(f"❌ tesseract : {e} (vérifie l'installation du binaire système)")

# --- docTR ---
try:
    from doctr.models import ocr_predictor
    ocr_engines["doctr"] = ocr_predictor(pretrained=True)
    print("✅ doctr chargé")
except Exception as e:
    echecs_ocr["doctr"] = str(e)
    print(f"❌ doctr : {e}")

print(f"\n{len(ocr_engines)}/{len(MODELES_OCR)} moteurs OCR prêts.")


In [ ]:
def _ocr_paddleocr(image_path):
    resultats = ocr_engines["paddleocr"].predict(str(image_path))
    lignes = []
    for page in resultats:
        textes = page.get("rec_texts", []) if isinstance(page, dict) else []
        lignes.extend(textes)
    return " ".join(lignes)


def _ocr_easyocr(image_path):
    resultats = ocr_engines["easyocr"].readtext(str(image_path), detail=0)
    return " ".join(resultats)


def _ocr_tesseract(image_path):
    from PIL import Image
    return ocr_engines["tesseract"].image_to_string(Image.open(image_path), lang="fra")


def _ocr_doctr(image_path):
    from doctr.io import DocumentFile
    doc = DocumentFile.from_images(str(image_path))
    resultat = ocr_engines["doctr"](doc)
    mots = []
    for page in resultat.pages:
        for bloc in page.blocks:
            for ligne in bloc.lines:
                for mot in ligne.words:
                    mots.append(mot.value)
    return " ".join(mots)


DISPATCH_OCR = {
    "paddleocr": _ocr_paddleocr, "easyocr": _ocr_easyocr,
    "tesseract": _ocr_tesseract, "doctr": _ocr_doctr,
}


def extraire_texte(nom_modele, image_path):
    """Retourne (texte, temps_ms, erreur)."""
    if nom_modele not in ocr_engines:
        return None, None, f"Modèle '{nom_modele}' non chargé"
    t0 = time.perf_counter()
    try:
        texte = DISPATCH_OCR[nom_modele](image_path)
        return texte.strip(), (time.perf_counter() - t0) * 1000, None
    except Exception as e:
        return None, (time.perf_counter() - t0) * 1000, str(e)


print("Dispatcher OCR prêt.")


## 3. Conversion PDF → images (si besoin)

In [ ]:
def pdf_vers_images(chemin_pdf, dossier_sortie=DATA_DIR, echelle=2):
    import pypdfium2 as pdfium

    pdf = pdfium.PdfDocument(chemin_pdf)
    chemins_images = []
    for i in range(len(pdf)):
        page = pdf[i]
        image = page.render(scale=echelle).to_pil()
        chemin = Path(dossier_sortie) / f"{Path(chemin_pdf).stem}_page{i+1}.png"
        image.save(chemin)
        chemins_images.append(chemin)
    return chemins_images


print("Fonction pdf_vers_images() prête.")


## 4. Jeu de test OCR (image + texte de référence corrigé manuellement)

Structure attendue : `data_pipeline/{id}.png` + une ligne dans `jeu_test_ocr` avec le texte exact du document.

In [ ]:
jeu_test_ocr = pd.DataFrame([
    {"id": "doc001", "fichier": str(DATA_DIR / "doc001.png"), "texte_reference": "Je veux aller à Dakar."},
    {"id": "doc002", "fichier": str(DATA_DIR / "doc002.png"), "texte_reference": "Comment vas-tu ?"},
    {"id": "doc003", "fichier": str(DATA_DIR / "doc003.png"), "texte_reference": "Veuillez fournir votre document."},
])

jeu_test_ocr["fichier_existe"] = jeu_test_ocr["fichier"].apply(lambda p: Path(p).exists())
if not jeu_test_ocr["fichier_existe"].any():
    print(f"⚠️  Aucune image trouvée dans {DATA_DIR}/ — dépose tes documents scannés (photos nettes, floues, inclinées, PDF...) avant de lancer le benchmark OCR.")

display(jeu_test_ocr)


## 5. Benchmark OCR : CER, WER, Precision, Recall, F1, latence

In [ ]:
try:
    from jiwer import wer as jiwer_wer, cer as jiwer_cer
    JIWER_OK = True
except ImportError:
    JIWER_OK = False
    print("⚠️  jiwer non installé -> pip install jiwer")


def normaliser_texte(texte: str) -> str:
    texte = str(texte).lower().strip()
    texte = unicodedata.normalize("NFKC", texte)
    texte = re.sub(r"\s+", " ", texte)
    return texte


def precision_recall_f1_mots(reference: str, hypothese: str):
    """Precision/Recall/F1 au niveau des mots (ensemble, ordre non pris en compte)."""
    mots_ref = set(normaliser_texte(reference).split())
    mots_hyp = set(normaliser_texte(hypothese).split())
    if not mots_hyp:
        return 0.0, 0.0, 0.0
    if not mots_ref:
        return 0.0, 0.0, 0.0
    vrais_positifs = len(mots_ref & mots_hyp)
    precision = vrais_positifs / len(mots_hyp)
    rappel = vrais_positifs / len(mots_ref)
    f1 = 2 * precision * rappel / (precision + rappel) if (precision + rappel) > 0 else 0.0
    return round(precision, 3), round(rappel, 3), round(f1, 3)


resultats_ocr = []
for _, ligne in jeu_test_ocr[jeu_test_ocr["fichier_existe"]].iterrows():
    for nom_modele in MODELES_OCR:
        texte_extrait, temps_ms, erreur = extraire_texte(nom_modele, ligne["fichier"])

        if erreur is not None or texte_extrait is None:
            resultats_ocr.append({
                "id": ligne["id"], "modele": nom_modele, "texte_extrait": None,
                "cer": None, "wer": None, "precision": None, "rappel": None, "f1": None,
                "latence_ms": round(temps_ms, 1) if temps_ms else None, "echec": True, "erreur": erreur,
            })
            continue

        cer = jiwer_cer(normaliser_texte(ligne["texte_reference"]), normaliser_texte(texte_extrait)) if JIWER_OK else None
        wer = jiwer_wer(normaliser_texte(ligne["texte_reference"]), normaliser_texte(texte_extrait)) if JIWER_OK else None
        precision, rappel, f1 = precision_recall_f1_mots(ligne["texte_reference"], texte_extrait)

        resultats_ocr.append({
            "id": ligne["id"], "modele": nom_modele, "texte_extrait": texte_extrait,
            "cer": round(cer, 4) if cer is not None else None,
            "wer": round(wer, 4) if wer is not None else None,
            "precision": precision, "rappel": rappel, "f1": f1,
            "latence_ms": round(temps_ms, 1), "echec": False, "erreur": None,
        })

df_ocr = pd.DataFrame(resultats_ocr)

if df_ocr.empty:
    print("⚠️  Aucun résultat OCR — vérifie que des images existent dans data_pipeline/.")
else:
    display(df_ocr[["id", "modele", "cer", "wer", "precision", "rappel", "f1", "latence_ms", "echec"]])


In [ ]:
if not df_ocr.empty:
    df_ocr_resume = df_ocr.groupby("modele")[["cer", "wer", "precision", "rappel", "f1", "latence_ms"]].mean(numeric_only=True).round(4)
    df_ocr_resume["taux_echec"] = df_ocr.groupby("modele")["echec"].mean().round(3)
    df_ocr_resume.to_csv(RESULTS_DIR / "ocr_resume.csv")
    print("=== Résumé OCR par modèle ===")
    display(df_ocr_resume)


## 6. Étage 2 — Traduction Français → Wolof : chargement et benchmark

Mêmes 3 modèles NLLB que le benchmark de traduction précédent — code condensé ici, réutilisable indépendamment.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizers_trad, modeles_trad, echecs_trad = {}, {}, {}

for nom, chemin in MODELES_FR_WO.items():
    try:
        tokenizers_trad[nom] = AutoTokenizer.from_pretrained(chemin)
        modeles_trad[nom] = AutoModelForSeq2SeqLM.from_pretrained(chemin).to(DEVICE)
        print(f"✅ {nom} chargé")
    except Exception as e:
        echecs_trad[nom] = str(e)
        print(f"❌ {nom} : {e}")

CODES_NLLB = {"fr": "fra_Latn", "wo": "wol_Latn"}


def traduire(nom_modele, texte, src="fr", tgt="wo", max_new_tokens=128):
    if nom_modele not in modeles_trad:
        return None, None, f"Modèle '{nom_modele}' non chargé"
    tokenizer = tokenizers_trad[nom_modele]
    modele = modeles_trad[nom_modele]
    t0 = time.perf_counter()
    try:
        tokenizer.src_lang = CODES_NLLB[src]
        inputs = tokenizer(texte, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        sortie = modele.generate(
            **inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids(CODES_NLLB[tgt]),
            max_new_tokens=max_new_tokens,
        )
        traduction = tokenizer.batch_decode(sortie, skip_special_tokens=True)[0]
        return traduction, (time.perf_counter() - t0) * 1000, None
    except Exception as e:
        return None, (time.perf_counter() - t0) * 1000, str(e)


print(f"\n{len(modeles_trad)}/{len(MODELES_FR_WO)} modèles de traduction prêts.")


In [ ]:
jeu_test_traduction = pd.DataFrame([
    {"id": "t001", "texte_francais": "Je veux aller à Dakar.", "reference_wolof": "Maa ngi bëgg dem Dakar."},
    {"id": "t002", "texte_francais": "Comment vas-tu ?", "reference_wolof": "Naka nga def ?"},
    {"id": "t003", "texte_francais": "Veuillez fournir votre document.", "reference_wolof": "Joxal ndigal bi."},
    {"id": "t004", "texte_francais": "Où se trouve le service de radiologie ?", "reference_wolof": "Fan la service radiologie ne?"},
    {"id": "t005", "texte_francais": "Merci beaucoup pour l'aide.", "reference_wolof": "Jërejëf lool ci dimbal bi."},
])

try:
    import sacrebleu
    SACREBLEU_OK = True
except ImportError:
    SACREBLEU_OK = False
    print("⚠️  sacrebleu non installé -> pip install sacrebleu")

resultats_trad = []
for _, ligne in jeu_test_traduction.iterrows():
    for nom_modele in MODELES_FR_WO:
        traduction, temps_ms, erreur = traduire(nom_modele, ligne["texte_francais"])

        bleu = chrf = ter = None
        if traduction and SACREBLEU_OK:
            bleu = round(sacrebleu.sentence_bleu(traduction, [ligne["reference_wolof"]]).score, 2)
            chrf = round(sacrebleu.sentence_chrf(traduction, [ligne["reference_wolof"]]).score, 2)
            ter = round(sacrebleu.sentence_ter(traduction, [ligne["reference_wolof"]]).score, 2)

        resultats_trad.append({
            "id": ligne["id"], "modele": nom_modele, "texte_francais": ligne["texte_francais"],
            "traduction_wolof": traduction, "reference_wolof": ligne["reference_wolof"],
            "bleu": bleu, "chrf": chrf, "ter": ter,
            "latence_ms": round(temps_ms, 1) if temps_ms else None,
            "echec": erreur is not None, "erreur": erreur,
        })

df_trad = pd.DataFrame(resultats_trad)
display(df_trad[["id", "modele", "traduction_wolof", "bleu", "chrf", "ter", "echec"]])

df_trad_resume = df_trad.groupby("modele")[["bleu", "chrf", "ter", "latence_ms"]].mean(numeric_only=True).round(3)
df_trad_resume["taux_echec"] = df_trad.groupby("modele")["echec"].mean().round(3)
df_trad_resume.to_csv(RESULTS_DIR / "traduction_resume.csv")
print("\n=== Résumé traduction par modèle ===")
display(df_trad_resume)


## 7. (Optionnel) COMET pour la traduction

Protégé par `try/except` — nécessite `unbabel-comet` et le téléchargement d'un modèle de plusieurs centaines de Mo.

In [ ]:
COMET_OK = False
try:
    from comet import download_model, load_from_checkpoint

    chemin_comet = download_model("Unbabel/wmt22-comet-da")
    modele_comet = load_from_checkpoint(chemin_comet)

    donnees_comet = [
        {"src": row["texte_francais"], "mt": row["traduction_wolof"], "ref": row["reference_wolof"]}
        for _, row in df_trad.iterrows() if row["traduction_wolof"]
    ]
    sortie_comet = modele_comet.predict(donnees_comet, batch_size=8, gpu_nb=1 if DEVICE == "cuda" else 0)

    idx_valides = df_trad[df_trad["traduction_wolof"].notna()].index
    df_trad.loc[idx_valides, "comet"] = [round(s, 4) for s in sortie_comet["scores"]]
    COMET_OK = True
    print("✅ COMET calculé.")
except Exception as e:
    print(f"⚠️  COMET non exécuté ({e}) -> pip install unbabel-comet")
    df_trad["comet"] = None


## 8. Étage 3 — TTS Wolof : chargement et benchmark (MOS auto, WER audio, SNR, RTF)

In [ ]:
# --- Chargement ASR pour le WER audio du TTS ---
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline

asr_pipeline = None
try:
    from peft import PeftModel
    base_model = WhisperForConditionalGeneration.from_pretrained(ASR_BASE_MODEL)
    model_asr = PeftModel.from_pretrained(base_model, ASR_MODEL_PATH).to(DEVICE)
    processor_asr = WhisperProcessor.from_pretrained(ASR_BASE_MODEL)
    asr_pipeline = pipeline(
        "automatic-speech-recognition", model=model_asr,
        tokenizer=processor_asr.tokenizer, feature_extractor=processor_asr.feature_extractor,
        device=0 if DEVICE == "cuda" else -1,
    )
    print("✅ ASR (LoRA) chargé pour le WER audio.")
except Exception as e_lora:
    try:
        asr_pipeline = pipeline("automatic-speech-recognition", model=ASR_MODEL_PATH, device=0 if DEVICE == "cuda" else -1)
        print("✅ ASR (modèle complet) chargé.")
    except Exception as e_full:
        print(f"❌ ASR indisponible ({e_full}) — le WER audio du TTS ne sera pas calculé.")


In [ ]:
tts_engines = {}
echecs_tts = {}

try:
    from TTS.api import TTS as CoquiTTS
    tts_engines["xtts_wolof"] = CoquiTTS(MODELES_TTS["xtts_wolof"]).to(DEVICE)
    print("✅ xtts_wolof chargé")
except Exception as e:
    echecs_tts["xtts_wolof"] = str(e)
    print(f"❌ xtts_wolof : {e}")

try:
    from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
    from datasets import load_dataset

    proc_st5 = SpeechT5Processor.from_pretrained(MODELES_TTS["speecht5_wolof"])
    model_st5 = SpeechT5ForTextToSpeech.from_pretrained(MODELES_TTS["speecht5_wolof"]).to(DEVICE)
    vocoder_st5 = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(DEVICE)
    try:
        embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
        speaker_embedding_st5 = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0).to(DEVICE)
    except Exception:
        speaker_embedding_st5 = torch.randn(1, 512).to(DEVICE)
    tts_engines["speecht5_wolof"] = {"processor": proc_st5, "model": model_st5, "vocoder": vocoder_st5, "speaker_embedding": speaker_embedding_st5}
    print("✅ speecht5_wolof chargé")
except Exception as e:
    echecs_tts["speecht5_wolof"] = str(e)
    print(f"❌ speecht5_wolof : {e}")

try:
    oolel_pipeline = pipeline("text-to-speech", model=MODELES_TTS["oolel_voices"], device=0 if DEVICE == "cuda" else -1)
    tts_engines["oolel_voices"] = oolel_pipeline
    print("✅ oolel_voices chargé")
except Exception as e:
    echecs_tts["oolel_voices"] = str(e)
    print(f"❌ oolel_voices : {e}")

print(f"\n{len(tts_engines)}/{len(MODELES_TTS)} moteurs TTS prêts.")


In [ ]:
def _synth_xtts(texte, speaker_wav=None, language="fr"):
    engine = tts_engines["xtts_wolof"]
    if speaker_wav is None:
        candidats = list(Path("reference_audio").glob("*.wav")) if Path("reference_audio").exists() else []
        speaker_wav = str(candidats[0]) if candidats else None
    if speaker_wav is None:
        raise ValueError("XTTS nécessite un audio de référence (reference_audio/*.wav)")
    wav = engine.tts(text=texte, speaker_wav=speaker_wav, language=language)
    return np.array(wav, dtype=np.float32), engine.synthesizer.output_sample_rate


def _synth_speecht5(texte):
    engine = tts_engines["speecht5_wolof"]
    inputs = engine["processor"](text=texte, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        speech = engine["model"].generate_speech(inputs["input_ids"], engine["speaker_embedding"], vocoder=engine["vocoder"])
    return speech.cpu().numpy(), 16000


def _synth_oolel(texte):
    sortie = tts_engines["oolel_voices"](texte)
    audio = np.array(sortie["audio"], dtype=np.float32).squeeze()
    return audio, sortie.get("sampling_rate", 16000)


DISPATCH_TTS = {"xtts_wolof": _synth_xtts, "speecht5_wolof": _synth_speecht5, "oolel_voices": _synth_oolel}


def synthetiser(nom_modele, texte):
    if nom_modele not in tts_engines:
        return None, None, None, f"Modèle '{nom_modele}' non chargé"
    t0 = time.perf_counter()
    try:
        audio, sr = DISPATCH_TTS[nom_modele](texte)
        return audio, sr, (time.perf_counter() - t0) * 1000, None
    except Exception as e:
        return None, None, (time.perf_counter() - t0) * 1000, str(e)


def estimer_snr(audio, sr, taille_frame_ms=25):
    taille_frame = int(sr * taille_frame_ms / 1000)
    n_frames = len(audio) // taille_frame
    if n_frames < 4:
        return None
    energies = np.array([np.mean(audio[i*taille_frame:(i+1)*taille_frame] ** 2) for i in range(n_frames)])
    energies = energies[energies > 0]
    if len(energies) < 4:
        return None
    bruit = energies[energies <= np.percentile(energies, 15)]
    signal = energies[energies >= np.percentile(energies, 85)]
    if len(bruit) == 0 or len(signal) == 0 or np.mean(bruit) == 0:
        return None
    return round(float(10 * np.log10(np.mean(signal) / np.mean(bruit))), 2)


print("Dispatcher TTS et fonctions de métriques prêts.")


In [ ]:
resultats_tts = []
for _, ligne in jeu_test_traduction.iterrows():
    texte_wolof = ligne["reference_wolof"]  # on synthétise le wolof de référence pour isoler la qualité TTS
    for nom_modele in MODELES_TTS:
        audio, sr, temps_ms, erreur = synthetiser(nom_modele, texte_wolof)

        chemin_audio, duree_s, rtf, wer_audio, snr = None, None, None, None, None
        if audio is not None:
            duree_s = len(audio) / sr
            rtf = round((temps_ms / 1000) / duree_s, 3) if duree_s > 0 else None
            chemin_audio = AUDIO_DIR / f"{nom_modele}_{ligne['id']}.wav"
            sf.write(chemin_audio, audio, sr)

            snr = estimer_snr(audio, sr)

            if asr_pipeline is not None:
                try:
                    texte_reconnu = asr_pipeline(str(chemin_audio), generate_kwargs={"task": "transcribe"})["text"].strip()
                    wer_audio = round(jiwer_wer(normaliser_texte(texte_wolof), normaliser_texte(texte_reconnu)), 4) if JIWER_OK else None
                except Exception:
                    wer_audio = None

        resultats_tts.append({
            "id": ligne["id"], "modele": nom_modele, "texte_wolof": texte_wolof,
            "audio_path": str(chemin_audio) if chemin_audio else None,
            "duree_audio_s": round(duree_s, 2) if duree_s else None,
            "latence_ms": round(temps_ms, 1) if temps_ms else None,
            "rtf": rtf, "wer_audio": wer_audio, "snr_db": snr,
            "echec": erreur is not None, "erreur": erreur,
        })

df_tts = pd.DataFrame(resultats_tts)
display(df_tts[["id", "modele", "wer_audio", "snr_db", "rtf", "latence_ms", "echec"]])

df_tts_resume = df_tts.groupby("modele")[["wer_audio", "snr_db", "rtf", "latence_ms"]].mean(numeric_only=True).round(3)
df_tts_resume["taux_echec"] = df_tts.groupby("modele")["echec"].mean().round(3)
df_tts_resume.to_csv(RESULTS_DIR / "tts_resume.csv")
print("\n=== Résumé TTS par modèle ===")
display(df_tts_resume)


### MOS (évaluation humaine)

Le naturel de la voix reste une évaluation humaine — cette fonction permet de noter manuellement les audios générés à l'étape précédente.

In [ ]:
def evaluer_mos_manuel(df, echantillon=None):
    a_noter = df[df["audio_path"].notna()]
    if echantillon:
        a_noter = a_noter.sample(min(echantillon, len(a_noter)), random_state=42)
    notes = []
    for _, row in a_noter.iterrows():
        print(f"\n--- {row['id']} | {row['modele']} --- Fichier : {row['audio_path']}")
        note = input("MOS (1=très mauvais ... 5=excellent) : ")
        notes.append({"id": row["id"], "modele": row["modele"], "mos_humain": int(note) if note else None})
    return pd.DataFrame(notes)


# df_mos = evaluer_mos_manuel(df_tts, echantillon=6)
print("Fonction prête : evaluer_mos_manuel(df_tts, echantillon=6)")


## 9. Performance globale : la chaîne complète Image/PDF → Audio

Teste des combinaisons OCR × Traduction × TTS de bout en bout, en mesurant la latence totale, le taux de réussite, la RAM/VRAM et le débit — pas seulement chaque étage isolément.

In [ ]:
def get_ram_mb():
    return psutil.Process().memory_info().rss / 1024**2


def get_vram_mb():
    return torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else None


# Meilleur modèle par étage jusqu'ici (sert de base si MODE_RAPIDE)
meilleur_ocr = df_ocr_resume["wer"].idxmin() if not df_ocr.empty and df_ocr_resume["wer"].notna().any() else list(ocr_engines.keys())[0] if ocr_engines else None
meilleur_trad = df_trad_resume["chrf"].idxmax() if df_trad_resume["chrf"].notna().any() else list(modeles_trad.keys())[0] if modeles_trad else None
meilleur_tts = df_tts_resume["wer_audio"].idxmin() if df_tts_resume["wer_audio"].notna().any() else list(tts_engines.keys())[0] if tts_engines else None

if MODE_RAPIDE:
    combinaisons_ocr = [meilleur_ocr] if meilleur_ocr else list(ocr_engines.keys())[:1]
    combinaisons_trad = list(MODELES_FR_WO.keys())
    combinaisons_tts = list(MODELES_TTS.keys())
    print(f"MODE_RAPIDE activé : OCR fixé sur '{meilleur_ocr}', {len(combinaisons_trad)} traductions × {len(combinaisons_tts)} TTS testés.")
else:
    combinaisons_ocr = list(MODELES_OCR.keys())
    combinaisons_trad = list(MODELES_FR_WO.keys())
    combinaisons_tts = list(MODELES_TTS.keys())
    print(f"MODE_RAPIDE désactivé : {len(combinaisons_ocr)} × {len(combinaisons_trad)} × {len(combinaisons_tts)} combinaisons complètes.")

resultats_pipeline = []

document_test = jeu_test_ocr[jeu_test_ocr["fichier_existe"]].iloc[0] if not jeu_test_ocr.empty and jeu_test_ocr["fichier_existe"].any() else None

if document_test is None:
    print("\n⚠️  Pas de document OCR disponible — le test de bout en bout utilisera directement une phrase française fixe (sans passer par l'OCR réel).")
    texte_francais_source = "Comment vas-tu ?"
else:
    texte_francais_source = None

for nom_ocr in combinaisons_ocr:
    for nom_trad in combinaisons_trad:
        for nom_tts in combinaisons_tts:
            ram_avant = get_ram_mb()
            t_debut_total = time.perf_counter()
            echec_etape = None

            # --- OCR ---
            if document_test is not None and nom_ocr:
                texte_fr, t_ocr, err_ocr = extraire_texte(nom_ocr, document_test["fichier"])
                if err_ocr:
                    echec_etape = f"OCR: {err_ocr}"
            else:
                texte_fr, t_ocr, err_ocr = texte_francais_source, 0.0, None

            # --- Traduction ---
            if echec_etape is None:
                texte_wo, t_trad, err_trad = traduire(nom_trad, texte_fr or "")
                if err_trad:
                    echec_etape = f"Traduction: {err_trad}"
            else:
                texte_wo, t_trad = None, 0.0

            # --- TTS ---
            if echec_etape is None and texte_wo:
                audio, sr, t_tts, err_tts = synthetiser(nom_tts, texte_wo)
                if err_tts:
                    echec_etape = f"TTS: {err_tts}"
            else:
                audio, t_tts = None, 0.0

            temps_total_ms = (time.perf_counter() - t_debut_total) * 1000
            ram_apres = get_ram_mb()

            resultats_pipeline.append({
                "ocr": nom_ocr, "traduction": nom_trad, "tts": nom_tts,
                "texte_francais": texte_fr, "texte_wolof": texte_wo,
                "latence_ocr_ms": round(t_ocr, 1) if t_ocr else None,
                "latence_trad_ms": round(t_trad, 1) if t_trad else None,
                "latence_tts_ms": round(t_tts, 1) if t_tts else None,
                "latence_totale_ms": round(temps_total_ms, 1),
                "ram_delta_mb": round(ram_apres - ram_avant, 2),
                "reussite": echec_etape is None and audio is not None,
                "echec_detail": echec_etape,
            })

df_pipeline = pd.DataFrame(resultats_pipeline)
display(df_pipeline[["ocr", "traduction", "tts", "latence_totale_ms", "reussite", "echec_detail"]])


In [ ]:
vram_totale = get_vram_mb()

print("=== Performance globale ===")
print(f"Taux de réussite global      : {df_pipeline['reussite'].mean()*100:.0f}%")
print(f"Latence totale moyenne       : {df_pipeline['latence_totale_ms'].mean():.0f} ms")
print(f"RAM (delta moyen par requête) : {df_pipeline['ram_delta_mb'].mean():.1f} MB")
if vram_totale is not None:
    print(f"VRAM max allouée (CUDA)      : {vram_totale:.0f} MB")

debit_par_min = 60000 / df_pipeline['latence_totale_ms'].mean() if df_pipeline['latence_totale_ms'].mean() > 0 else None
if debit_par_min:
    print(f"Débit estimé                 : {debit_par_min:.1f} documents/minute")

chemin_pipeline = RESULTS_DIR / "pipeline_resultats_complets.csv"
df_pipeline.to_csv(chemin_pipeline, index=False)
print(f"\nRésultats de bout en bout sauvegardés : {chemin_pipeline}")


## 10. Tableau de benchmark final combiné

Fusionne les métriques par étage avec la performance globale, pour chaque combinaison testée.

In [ ]:
df_final = df_pipeline[df_pipeline["reussite"]].copy()

# Rattacher les métriques d'étage (moyennes par modèle, faute de pouvoir les recalculer par combinaison exacte)
df_final = df_final.merge(
    df_ocr_resume[["cer", "wer"]].add_prefix("ocr_"), left_on="ocr", right_index=True, how="left",
) if not df_ocr_resume.empty else df_final

df_final = df_final.merge(
    df_trad_resume[["bleu", "chrf", "ter"]].add_prefix("trad_"), left_on="traduction", right_index=True, how="left",
)

df_final = df_final.merge(
    df_tts_resume[["wer_audio", "snr_db"]].add_prefix("tts_"), left_on="tts", right_index=True, how="left",
)

colonnes_affichage = [
    "ocr", "traduction", "tts",
    "ocr_cer", "trad_chrf", "trad_ter", "tts_wer_audio", "latence_totale_ms",
]
colonnes_affichage = [c for c in colonnes_affichage if c in df_final.columns]

df_final.to_csv(RESULTS_DIR / "tableau_benchmark_final.csv", index=False)
display(df_final[colonnes_affichage].sort_values("latence_totale_ms"))


## 11. Sélection et sauvegarde du meilleur pipeline

Score composite : OCR CER faible + traduction chrF élevé/TER faible + TTS WER audio faible + latence totale acceptable. Le meilleur triplet (OCR, traduction, TTS) est identifié, puis les modèles de traduction et TTS gagnants sont sauvegardés sur disque.

In [ ]:
def normaliser(serie, inverser=False):
    s = serie.dropna()
    if s.empty or s.max() == s.min():
        return pd.Series(0.5, index=serie.index)
    norm = (serie - s.min()) / (s.max() - s.min())
    return 1 - norm if inverser else norm


ponderations = [
    ("ocr_cer", 0.15, True),
    ("trad_chrf", 0.20, False),
    ("trad_ter", 0.10, True),
    ("tts_wer_audio", 0.25, True),
    ("latence_totale_ms", 0.15, True),
]

if not df_final.empty:
    score = pd.Series(0.0, index=df_final.index)
    poids_total = pd.Series(0.0, index=df_final.index)

    for col, poids, inverser in ponderations:
        if col not in df_final.columns:
            continue
        dispo = df_final[col].notna()
        if not dispo.any():
            continue
        score += normaliser(df_final[col], inverser=inverser).fillna(0) * poids
        poids_total += dispo.astype(float) * poids

    df_final["score_composite"] = (score / poids_total.replace(0, np.nan)).round(4)
    df_final_trie = df_final.sort_values("score_composite", ascending=False)
    display(df_final_trie[["ocr", "traduction", "tts", "score_composite", "latence_totale_ms"]])

    meilleure_combo = df_final_trie.iloc[0]
    print(f"\n🏆 MEILLEUR PIPELINE : OCR={meilleure_combo['ocr']} + Traduction={meilleure_combo['traduction']} + TTS={meilleure_combo['tts']}")
    print(f"   Score composite = {meilleure_combo['score_composite']:.3f} | Latence totale = {meilleure_combo['latence_totale_ms']:.0f} ms")
else:
    print("⚠️ Aucune combinaison réussie à classer.")


In [ ]:
from huggingface_hub import snapshot_download
import json as _json

if not df_final.empty:
    MEILLEUR_PIPELINE_DIR.mkdir(exist_ok=True)
    meilleure_combo = df_final.sort_values("score_composite", ascending=False).iloc[0]

    # Sauvegarde du meilleur modèle de traduction
    try:
        snapshot_download(repo_id=MODELES_FR_WO[meilleure_combo["traduction"]], local_dir=MEILLEUR_PIPELINE_DIR / "traduction")
        print(f"✅ Modèle de traduction '{meilleure_combo['traduction']}' sauvegardé.")
    except Exception as e:
        print(f"❌ Échec sauvegarde traduction : {e}")

    # Sauvegarde du meilleur modèle TTS
    try:
        snapshot_download(repo_id=MODELES_TTS[meilleure_combo["tts"]], local_dir=MEILLEUR_PIPELINE_DIR / "tts")
        print(f"✅ Modèle TTS '{meilleure_combo['tts']}' sauvegardé.")
    except Exception as e:
        print(f"❌ Échec sauvegarde TTS : {e}")

    fiche = {
        "pipeline_gagnant": {
            "ocr": meilleure_combo["ocr"],
            "traduction": MODELES_FR_WO[meilleure_combo["traduction"]],
            "tts": MODELES_TTS[meilleure_combo["tts"]],
        },
        "score_composite": float(meilleure_combo["score_composite"]),
        "latence_totale_ms": float(meilleure_combo["latence_totale_ms"]),
        "date_benchmark": pd.Timestamp.now().isoformat(),
    }
    with open(MEILLEUR_PIPELINE_DIR / "fiche_pipeline.json", "w", encoding="utf-8") as f:
        _json.dump(fiche, f, ensure_ascii=False, indent=2)

    print(f"\nFiche du pipeline sauvegardée : {MEILLEUR_PIPELINE_DIR / 'fiche_pipeline.json'}")
    print("(Le moteur OCR retenu reste à charger via sa bibliothèque native — PaddleOCR/EasyOCR/Tesseract/docTR ne se 'sauvegardent' pas de la même façon que les modèles Hugging Face.)")


## 12. Graphiques comparatifs

In [ ]:
import matplotlib.pyplot as plt

if not df_final.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    df_ocr_resume["cer"].plot(kind="bar", ax=axes[0], color="#4C72B0", title="OCR — CER moyen (plus bas = mieux)")
    axes[0].tick_params(axis="x", rotation=25)

    df_trad_resume["chrf"].plot(kind="bar", ax=axes[1], color="#DD8452", title="Traduction — chrF moyen (plus haut = mieux)")
    axes[1].tick_params(axis="x", rotation=25)

    df_tts_resume["wer_audio"].plot(kind="bar", ax=axes[2], color="#55A868", title="TTS — WER audio moyen (plus bas = mieux)")
    axes[2].tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "graphiques_par_etage.png", dpi=150)
    plt.show()

    plt.figure(figsize=(8, 6))
    plt.scatter(df_final["latence_totale_ms"], df_final["score_composite"],
                s=120, c=["#4C72B0", "#DD8452", "#55A868", "#C44E52"] * (len(df_final)//4 + 1))
    for _, row in df_final.iterrows():
        label = f"{row['ocr']}+{row['traduction']}+{row['tts']}"
        plt.annotate(label, (row["latence_totale_ms"], row["score_composite"]), fontsize=7, xytext=(5, 5), textcoords="offset points")
    plt.xlabel("Latence totale (ms)")
    plt.ylabel("Score composite (plus haut = mieux)")
    plt.title("Compromis qualité / rapidité par pipeline (idéal = en haut à gauche)")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "graphique_compromis_pipeline.png", dpi=150)
    plt.show()


## 13. Bilan final

In [ ]:
print("=== BILAN DU BENCHMARK COMPLET ===\n")

print("--- Étage OCR ---")
if not df_ocr.empty:
    print(f"Meilleur CER  : {df_ocr_resume['cer'].idxmin()} ({df_ocr_resume['cer'].min():.4f})")
    print(f"Plus rapide   : {df_ocr_resume['latence_ms'].idxmin()} ({df_ocr_resume['latence_ms'].min():.0f} ms)")

print("\n--- Étage Traduction ---")
print(f"Meilleur chrF : {df_trad_resume['chrf'].idxmax()} ({df_trad_resume['chrf'].max():.2f})")
print(f"Meilleur TER  : {df_trad_resume['ter'].idxmin()} ({df_trad_resume['ter'].min():.2f})")

print("\n--- Étage TTS ---")
if df_tts_resume["wer_audio"].notna().any():
    print(f"Meilleur WER audio : {df_tts_resume['wer_audio'].idxmin()} ({df_tts_resume['wer_audio'].min():.4f})")
if df_tts_resume["snr_db"].notna().any():
    print(f"Meilleur SNR       : {df_tts_resume['snr_db'].idxmax()} ({df_tts_resume['snr_db'].max():.1f} dB)")

print("\n--- Pipeline complet ---")
if not df_final.empty:
    meilleure_combo = df_final.sort_values('score_composite', ascending=False).iloc[0]
    print(f"🏆 {meilleure_combo['ocr']} + {meilleure_combo['traduction']} + {meilleure_combo['tts']}")
    print(f"   Score composite : {meilleure_combo['score_composite']:.3f}")
    print(f"   Latence totale  : {meilleure_combo['latence_totale_ms']:.0f} ms")
    print(f"   Taux de réussite global : {df_pipeline['reussite'].mean()*100:.0f}%")

print("\n--- Limites de ce run ---")
print("- MOS humain non renseigné tant que evaluer_mos_manuel() n'a pas été exécuté")
print("- Jeu de test réduit (3 documents OCR, 5 phrases) — élargis-le pour un résultat scientifiquement robuste")
print("- PESQ / similarité du locuteur / corrélation de pitch non inclus ici (voir le notebook TTS dédié)")
if echecs_ocr:
    print(f"- OCR non chargés : {list(echecs_ocr.keys())}")
if echecs_trad:
    print(f"- Traduction non chargée : {list(echecs_trad.keys())}")
if echecs_tts:
    print(f"- TTS non chargés : {list(echecs_tts.keys())}")
